[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SAS-DHRH/wb-wos/blob/main/notebooks/VGG-WISE/image_search.ipynb)

# Demo of Image Search using WISE

Welcome! This notebook introduces **[WISE](https://www.robots.ox.ac.uk/~vgg/software/wise/)**, an open-source search engine that lets you search image collections using natural language and example images as search query.

In the next few cells you will:

1. Install WISE
2. Choose a small demo dataset
3. Extract image features using a vision-language model
4. Build a search index
5. Search the collection interactively in your browser

No programming experience is required — just run each cell in order by clicking the run button on its left (or pressing `Shift`+`Enter`).

> For the best experience, we recommend using the Google Chrome browser and signing in to your Google account. Some features of this notebook may not work in other browsers or without being signed in.
>
> The first run takes a few minutes. Installing WISE and downloading the vision language model (`ViT-B-16-SigLIP2-512`, a few hundred MB) happens only once per Colab session.
>
> A GPU makes this much faster. This notebook requests a GPU automatically. If you ever need to set it manually, go to **Runtime > Change runtime type > Hardware accelerator > GPU**.

---

Created by Abhishek Dutta (adutta@robots.ox.ac.uk), Visual Geometry Group, University of Oxford.


## Troubleshooting

**The search interface in Step 5 does not appear**

This is usually a browser compatibility issue. Try the following in order:

1. Reopen this notebook in [Google Chrome](https://www.google.com/chrome/),
   make sure you are signed in to your Google account, and re-run Step 5.
   If the session is still active, the server will still be running and
   you won't need to re-run the earlier steps.
2. If Step 5 still doesn't show the interface, or if too much time has passed
   since you last ran the notebook, re-run all steps from the beginning.

**The search interface shows "unable to load search results"**

This is an occasional hiccup with the connection between your browser and the
notebook. It's not a problem with your images or the search index. Simply try
your search again and it should go through.

**The search interface was working but has stopped responding**

The notebook may have disconnected in the background. Check the connection
indicator in the top-right corner of the Colab page. If it shows as
disconnected, reconnect and re-run Steps 1-5 to restore the working area and
restart the server.

## Step 1 — Install WISE

This downloads the WISE software, installs its Python dependencies, and builds the web-based search interface. This is the longest step — please be patient (takes around 3 to 5 minutes).

> When you run the cell below, you may see some pip dependency conflicts and deprecation warnings. They are safe to ignore.


In [ ]:
#@title Install WISE (run once) { display-mode: "form" }
%cd /content

# 1. Download WISE source code
!rm -rf wise
!git clone --quiet --depth 1 https://gitlab.com/vgg/wise/wise.git
!cd wise && git checkout 499f1cf5c922fa1e1b53e9fe182bf90958412fef

# 2. Install WISE and its Python dependencies
#    (uses the CPU build of FAISS, which is robust and plenty fast for small demos)
!pip install -q -r wise/requirements.txt
!pip install -q wise/

# 3. Build the web-based search interface (React + Ant Design)
!cd wise/frontend && npm install --silent && npm run build

%cd /content/wise

# This notebook is set to use a GPU automatically. Check that one is available.
import torch
if torch.cuda.is_available():
    print(f"\nWISE installed. GPU is available ({torch.cuda.get_device_name(0)}).")
else:
    print("\nWISE installed, but NO GPU was detected. The notebook will still work, "
          "but will be much slower.\nTo enable a GPU, go to "
          "Runtime > Change runtime type > Hardware accelerator > GPU, then re-run this cell.")


## Step 2 — Select images from your Google Drive

Upload your images to your Google Drive either as a `.zip` archive or a folder of images, and then enter the path to either the zip file or folder in the `GDRIVE_PATH` field.

Google Drive will be mounted automatically. Any image larger than 800 pixels
is resized (keeping its aspect ratio) and converted to JPEG before indexing.
If you provide more than 1000 images, feature extraction may take a long time.

### Supported image formats

The following image formats are accepted: JPEG, PNG, TIFF, BMP, GIF, and WebP.
Any other file types inside your ZIP or folder will be skipped automatically.

All images are converted to JPEG before indexing. As part of this, any image
larger than 800 pixels (in either dimension) is resized, keeping its original
proportions.

**A note on TIFF files:** TIFF is common in archival and GLAM collections, and
scanners frequently produce 16-bit TIFFs. Standard JPEG conversion does not
handle 16-bit images correctly and would produce blank or washed-out results.
This notebook detects 16-bit TIFFs automatically and normalises them to 8-bit
before conversion, so colours and tones are preserved correctly without any
extra steps on your part.

### How to find your path

1. Open [Google Drive](https://drive.google.com) in a new browser tab.
2. Navigate to your ZIP file or image folder.
3. Note where it sits inside your Drive.

Examples:

(a) Your images are in a `.zip` archive

- If your file `my-images.zip` is in a folder called `wise` inside `My Drive`, your path would be: `wise/my-images.zip`.
   
- If the file sits directly inside `My Drive` (not in any folder), just enter the filename: `my-images.zip`.

(b) Your images are in a folder

- If your images are in a folder (not a zip file) `my-images` inside `My Drive`, your path would be: `wise/my-images`.

- If the folder sits directly inside `My Drive` (not in any folder), just enter the filename: `my-images`.

### What happens to your images

Your original files in Google Drive are never modified. When you run this cell,
this notebook copies your images into a temporary working area so they can be prepared for searching (resizing anything larger than 800 pixels and converting to JPEG).

**Important:** The temporary working area is wiped automatically when you close
or disconnect from this notebook. Your originals in Google Drive are always safe. If you come back to this notebook later, simply re-run Steps 2-4 to rebuild the working area. Step 5 (searching) is the only part that benefits from
picking up where you left off, and it will prompt you if anything is missing.

In [ ]:
#@title Choose your own images
GDRIVE_PATH = "" #@param {type:"string", placeholder:"Enter the path to images in your Google Drive"}

import os, io, shutil, zipfile
from pathlib import Path
from PIL import Image
from google.colab import drive

MAX_SIZE = 800
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tiff", ".tif", ".webp"}

def prepare_image(name, data, out_dir):
    """Resize large images and convert to JPEG. Returns True if saved."""
    import numpy as np

    try:
        img = Image.open(io.BytesIO(data))
    except Exception:
        print(f"  skipped (not a readable image): {name}")
        return False

    # Normalise 16-bit images to 8-bit before conversion.
    # PIL does not do this automatically, resulting in blank or washed-out
    # JPEGs, a common issue with TIFF files from scanners and GLAM collections.
    if img.mode in ("I;16", "I;16B", "I;16L", "I;16S"):
        # Straightforward 16-bit: scale from 0–65535 to 0–255
        arr = np.array(img, dtype=np.float32)
        arr = (arr / 65535.0 * 255).clip(0, 255).astype(np.uint8)
        img = Image.fromarray(arr, mode="L")
    elif img.mode == "I":
        # PIL sometimes loads 16-bit TIFFs as 32-bit signed integers,
        # normalise to the actual min/max range present in the image
        arr = np.array(img, dtype=np.float32)
        if arr.max() > arr.min():
            arr = (arr - arr.min()) / (arr.max() - arr.min()) * 255
        img = Image.fromarray(arr.clip(0, 255).astype(np.uint8), mode="L")

    img = img.convert("RGB")
    if max(img.size) > MAX_SIZE:
        img.thumbnail((MAX_SIZE, MAX_SIZE))
    img.save(os.path.join(out_dir, Path(name).stem + ".jpg"), "JPEG", quality=90)
    return True

# Mount Google Drive if not already mounted
if not os.path.isdir("/content/drive/MyDrive"):
    print("Mounting Google Drive...")
    drive.mount("/content/drive")

# Normalise path - accept "my-images.zip", "my-images/", or full "/content/drive/..." form
gdrive_path = GDRIVE_PATH.strip().rstrip("/")
if not gdrive_path:
    raise ValueError("Please enter a path in the GDRIVE_PATH field above.")
if not gdrive_path.startswith("/"):
    gdrive_path = f"/content/drive/MyDrive/{gdrive_path}"
if not os.path.exists(gdrive_path):
    raise FileNotFoundError(f"No file or folder found at: {gdrive_path}")

# Collect (name, bytes) pairs from either a ZIP or a folder
items = []

if os.path.isfile(gdrive_path):
    # ---- ZIP file ----
    if not gdrive_path.lower().endswith(".zip"):
        raise ValueError(f"Expected a .zip file or a folder, got: {gdrive_path}")
    source_name = Path(gdrive_path).stem
    print(f"Reading ZIP: {gdrive_path}")
    with zipfile.ZipFile(gdrive_path) as zf:
        for member in zf.namelist():
            if member.endswith("/") or os.path.basename(member).startswith("."):
                continue
            if Path(member).suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            items.append((member, zf.read(member)))

elif os.path.isdir(gdrive_path):
    # ---- Folder ----
    source_name = Path(gdrive_path).name
    print(f"Reading folder: {gdrive_path}")
    for p in sorted(Path(gdrive_path).rglob("*")):
        if not p.is_file():
            continue
        if p.name.startswith("."):
            continue
        if p.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        items.append((p.name, p.read_bytes()))

MEDIA_DIRECTORY   = f"/content/wise/wise-data/{source_name}"
PROJECT_DIRECTORY = f"/content/wise/wise-projects/{source_name}"

if os.path.isdir(MEDIA_DIRECTORY):
    shutil.rmtree(MEDIA_DIRECTORY)
os.makedirs(MEDIA_DIRECTORY, exist_ok=True)

if len(items) > 1000:
    print(f"\nWarning: {len(items)} images found. Feature extraction may take a long time.")

print(f"\nPreparing {len(items)} file(s) (resizing large images, converting to JPEG)...")
n_images = sum(prepare_image(name, data, MEDIA_DIRECTORY) for name, data in items)
print(f"\nReady — {n_images} images prepared in {MEDIA_DIRECTORY}")

## Step 3 — Extract image features

WISE passes every image through a vision-language model (`ViT-B-16-SigLIP2-512`). This turns each image into a numeric embedding (or vector) that captures its visual content, so we can later match images against text descriptions.

> The model is downloaded automatically the first time it is used — this may take a couple of minutes.


In [ ]:
#@title Extract features { display-mode: "form" }
VISION_MODEL = "mlfoundations/open_clip/ViT-B-16-SigLIP2-512/webli"

!python3 -m wise extract-features "{MEDIA_DIRECTORY}" \
    --project-dir "{PROJECT_DIRECTORY}" \
    --image-feature-id "{VISION_MODEL}" \
    -y


## Step 4 — Build the search index

We build a [FAISS](https://github.com/facebookresearch/faiss) index over the embeddings. This is what makes searching fast and accurate.


In [ ]:
#@title Create the search index { display-mode: "form" }
!python3 -m wise create-index \
    --project-dir "{PROJECT_DIRECTORY}" \
    --index-type IndexFlatIP


## Step 5 — Search

This starts the WISE web server and opens the search interface right here in the notebook.

Starting the server takes around 30 seconds. Please wait until you see the message **"WISE is running"** and the search interface appears below.

For the **15th century printed illustrations from Venice**, try searches like:
- a boy and a girl talking to each other
- camel
- aesop
- death
- congregation of people
- animals

For the **vase images from the Oxford Beazley Archive of Greek pottery**, try searches like:
- a warrior holding a shield and spear
- a horse
- two figures facing each other
- a winged figure
- a person playing a musical instrument

You can also use the image/upload icon in the interface to search by visual similarity with an example image.

> When you are finished, stop this cell (the stop button) to shut the server down.


In [ ]:
#@title Start the WISE search interface { display-mode: "form" }
DISPLAY_MODE = "Embedded in notebook" #@param ["Embedded in notebook", "Browser tab"]

import subprocess, time, socket
from pathlib import Path
from google.colab.output import serve_kernel_port_as_iframe, serve_kernel_port_as_window

PORT = 9670

# serve runs a blocking web server, so we launch it in the background.
server = subprocess.Popen(["python3", "-m", "wise", "serve", "--project-dir", PROJECT_DIRECTORY])

# Wait until the server is ready to accept connections (includes model warm-up).
def wait_for_port(port, timeout=240):
    start = time.time()
    while time.time() - start < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(2)
    return False

if wait_for_port(PORT):
    print("WISE is running. The search interface or a link to open in a new tab will appear below.\n")
    if DISPLAY_MODE == "Browser tab":
        serve_kernel_port_as_window(PORT, path=f"/{Path(PROJECT_DIRECTORY).name}/")
    else:
        serve_kernel_port_as_iframe(PORT, path=f"/{Path(PROJECT_DIRECTORY).name}/")
else:
    print("The server did not start in time. Re-run this cell, or check the log output above.")


<br><br>

---

## Version history

- v1.0 (1 June 2026) : Initial release for the [ScholarThon](https://blog.humanities.org.uk/2026/05/14/scholarthon-rethinking-the-hackathon-for-arts-and-humanities-research/) event.
- v1.1 (3 June 2026, KK) : Derived from v1.0 for the "load images from Google Drive" use case. Changes: removed demo dataset and file upload options (Step 2 now loads directly from Google Drive); added 16-bit TIFF normalisation; added Troubleshooting section; fixed "No module named 'wise'" error (Step 1 now installs WISE as a package).
- v1.2 (16 June 2026, KK) : Rebased onto VGG's updated v1.0 notebook, which fixes broken CLI calls (standalone scripts replaced by `python3 -m wise`) and resolves the "No module named 'wise'" error. Carried forward from v1.1: Google Drive image loading (Step 2), Troubleshooting section, and version history. Added WISE version pinning to a specific commit hash (Step 1) for stability.
- v1.3 (16 June 2026, KK) : Added display mode selector to Step 5, allowing the WISE search interface to be opened either in a separate browser tab or embedded within the notebook. Browser tab is the recommended default as it provides more screen space.